## stellar parameters

In [62]:
import pandas as pd

df = pd.read_parquet('../../Zhang2023_lamost.parquet')
df.shape

(21919765, 9)

In [71]:
# ID = 5239758155778687360 #TOI-6715
ID = 1935439776865267072 #TOI-5973
df.gaia_dr3_id.isin([ID]).sum()

0

In [72]:
type(df['gaia_dr3_id'].iloc[0])

numpy.int64

In [73]:
df.query("gaia_dr3_id==@ID")

,teff_K,feh_dex,logg_dex,extinction_mag,parallax_mas,ra,dec,gaia_dr3_id,quality_flag


## download and merge

https://zenodo.org/records/7811871

In [1]:
import h5py

fp = "../data/Zhang2023/stellar_params_catalog_00.h5"

with h5py.File(fp, 'r') as f:
    print("Keys at the root level:")
    for key in f.keys():
        print(key)

Keys at the root level:
chi2_opt
dec
feh_confidence
gdr3_source_id
ln_prior
logg_confidence
quality_flags
ra
stellar_params_err
stellar_params_est
teff_confidence


In [2]:
with h5py.File(fp, 'r') as f:
    try:
        params = f['stellar_params_est'][()]
        err = f['stellar_params_err'][()]
        qf = f['quality_flags'][()]
        ra = f['ra'][()]
        dec = f['dec'][()]
        gid = f['gdr3_source_id'][()]
    except Exception as e:
        print(e)

In [3]:
import pandas as pd
#effective temperature in kiloKelvin, [Fe/H] in dex, log(g) in dex, E in mag, parallax in mas
col_names = ['teff_K','feh_dex','logg_dex','extinction_mag','parallax_mas']
df = pd.DataFrame(params, columns=col_names)
df['teff_K'] *= 1e3
df['ra'] = ra
df['dec'] = dec
df['gaia_dr3_id'] = gid
df['quality_flag'] = qf
df.head()

,teff_K,feh_dex,logg_dex,extinction_mag,parallax_mas,ra,dec,gaia_dr3_id,quality_flag
0,5372.870445,-0.438730,4.417336,0.074882,0.356730,44.996155,0.005615,4295806720,0
1,4720.390320,-0.208398,4.739237,0.076308,3.133679,45.004978,0.019880,38655544960,0
2,5458.602428,0.020589,4.474086,0.093338,0.645782,44.993271,0.076334,1275606125952,0
3,6705.673695,-0.763871,4.324942,0.200056,0.331891,44.996063,0.084918,1653563247744,2
4,5962.486744,-0.447542,4.329394,0.071844,2.674073,45.132145,0.137854,2851858288640,0


In [53]:
from glob import glob

h5_files = sorted(list(glob("../data/Zhang2023/stellar_params_catalog_*.h5")))
h5_files

['../data/Zhang2023/stellar_params_catalog_00.h5',
 '../data/Zhang2023/stellar_params_catalog_01.h5',
 '../data/Zhang2023/stellar_params_catalog_02.h5',
 '../data/Zhang2023/stellar_params_catalog_03.h5',
 '../data/Zhang2023/stellar_params_catalog_04.h5',
 '../data/Zhang2023/stellar_params_catalog_05.h5',
 '../data/Zhang2023/stellar_params_catalog_06.h5',
 '../data/Zhang2023/stellar_params_catalog_07.h5',
 '../data/Zhang2023/stellar_params_catalog_08.h5',
 '../data/Zhang2023/stellar_params_catalog_09.h5']

In [54]:
from astropy.table import Table
from tqdm import tqdm

for fp in tqdm(h5_files[:-1]):
    print(f"Loading {fp}")
    output = Table()
    with h5py.File(fp, 'r') as f:
        for i, key in enumerate(f.keys()):
            #print(f"Loading {i+1}/{len(f.keys())}: {key}")
            output[key] = f[key][:]

  0%|                                                                                                                                                                              | 0/9 [00:00<?, ?it/s]

Loading ../data/Zhang2023/stellar_params_catalog_00.h5


 11%|██████████████████▍                                                                                                                                                   | 1/9 [00:15<02:07, 15.98s/it]

Loading ../data/Zhang2023/stellar_params_catalog_01.h5


 22%|████████████████████████████████████▉                                                                                                                                 | 2/9 [00:31<01:50, 15.80s/it]

Loading ../data/Zhang2023/stellar_params_catalog_02.h5


... (9 outputs left out)

Loading ../data/Zhang2023/stellar_params_catalog_07.h5


 89%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▌                  | 8/9 [02:04<00:15, 15.27s/it]

Loading ../data/Zhang2023/stellar_params_catalog_08.h5


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 9/9 [02:19<00:00, 15.46s/it]


In [55]:
output['stellar_params_est'].value.shape

(21919765, 5)

In [56]:
import pandas as pd

#effective temperature in kiloKelvin, [Fe/H] in dex, log(g) in dex, E in mag, parallax in mas
col_names = ['teff_K','feh_dex','logg_dex','extinction_mag','parallax_mas']
df = pd.DataFrame(output['stellar_params_est'].value, columns=col_names)
df['teff_K'] *= 1e3
df['ra'] = output['ra'].value
df['dec'] = output['dec'].value
df['gaia_dr3_id'] = output['gdr3_source_id'].value
df['quality_flag'] = output['quality_flags'].value
df.head()

,teff_K,feh_dex,logg_dex,extinction_mag,parallax_mas,ra,dec,gaia_dr3_id,quality_flag
0,4839.326859,-0.099631,2.430617,1.979814,0.189892,220.084351,-57.457836,5891466769470359168,9
1,2447.894573,-0.526820,0.176255,1.437817,0.448885,220.095520,-57.448490,5891466773810259200,31
2,11052.608490,-0.194516,3.730735,0.629586,0.424890,220.089951,-57.447929,5891466773810260736,3
3,4435.482979,0.363395,2.710076,0.930054,0.347854,220.082047,-57.447243,5891466773810263040,8
4,5024.353981,0.277466,3.831023,1.109913,0.495563,220.094147,-57.444946,5891466773810263296,7


In [57]:
df.shape

(21919765, 9)

In [58]:
# ID = 5239758155778687360 #TOI-6715
ID = 1935439776865267072 #TOI-5973
df.gaia_dr3_id.isin([ID]).sum()

0

In [63]:
df.query("gaia_dr3_id==@ID")

,teff_K,feh_dex,logg_dex,extinction_mag,parallax_mas,ra,dec,gaia_dr3_id,quality_flag


In [59]:
df2 = df[df['quality_flag']<8]
df2.shape

(17126204, 9)

In [60]:
df.to_parquet('../../Zhang2023_lamost.parquet')

In [61]:
!ls -lth '../../Zhang2023_lamost.parquet'

-rw-r--r-- 1 jerome muscat 848M Nov 26 09:34 ../../Zhang2023_lamost.parquet


## lamost spectra

In [28]:
from astroquery.esasky import ESASky

spectra_list = ESASky.list_spectra()
spectra_list

['XMM-NEWTON',
 'Chandra',
 'IUE',
 'HST-UV',
 'HST-OPTICAL',
 'JWST-MID-IR',
 'JWST-NEAR-IR',
 'HST-IR',
 'ISO-IR',
 'Herschel',
 'LAMOST_LRS',
 'LAMOST_MRS',
 'CHEOPS']

In [32]:
result = ESASky.query_object_catalogs(position="TOI-5973", catalogs="LAMOST_MRS")
result

TableList with 1 tables:
	'0:LAMOST_MRS' with 57 column(s) and 8 row(s) 

In [51]:
result[0]

product_url,designation,obsid,ra,dec,uid,gp_id,mobsid,obsdate,band,lmjd,snr,mjd,rv_b0,planid,rv_r0,spid,fiberid,lmjm,ra_obs,dec_obs,gaia_source_id,gaia_g_mean_mag,gaia_bp_mean_mag,gaia_rp_mean_mag,tsource,fibertype,tfrom,tcomment,offsets,offsets_v,rv_b0_err,rv_b1,rv_b1_err,rv_b_flag,rv_r0_err,rv_r1,rv_r1_err,rv_r_flag,rv_br0,rv_br0_err,rv_br1,rv_br1_err,rv_br_flag,rv_lasp0,rv_lasp0_err,rv_lasp1,rv_lasp1_err,coadd,fibermask,bad_b,bad_r,moon_angle,lunardate,moon_flg,lamost_dr8_mrs_oid,npix
,,,degrees,degrees,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
object,object,int64,float64,float64,object,object,object,object,object,int32,float64,int32,float64,object,float64,int32,int32,int32,float64,float64,object,float64,float64,float64,object,object,object,object,bool,float64,float64,float64,float64,int32,float64,float64,float64,int32,float64,float64,float64,float64,int32,float64,float64,float64,float64,bool,int32,bool,bool,float64,int32,bool,int32,int64
http://www.lamost.org/dr8/v2.0//medspectrum/fits/681112046,J225014.39+433935.9,681112046,342.55997,43.659987,G14220639911232,1935439776865267072,68111204684108765B,2018-10-17T00:00:00.0,B,58409,26.76,58408,-12.86,NT224405N414909M01,-5.44,12,46,84108765,342.55997,43.659987,1935439776865267072,12.9221,13.2615,12.4196,MRS_NT_M,Obj,GAIA,-,False,0.0,1.15,-13.11,1.44,0,1.51,-7.57,1.8,0,-14.31,0.99,-15.2,1.08,0,-12.91,0.91,-12.87,0.95,False,0,False,False,70.0,9,False,5606422,220034
http://www.lamost.org/dr8/v2.0//medspectrum/fits/681112046,J225014.39+433935.9,681112046,342.55997,43.659987,G14220639911232,1935439776865267072,68111204684108765R,2018-10-17T00:00:00.0,R,58409,41.47,58408,-12.86,NT224405N414909M01,-5.44,12,46,84108765,342.55997,43.659987,1935439776865267072,12.9221,13.2615,12.4196,MRS_NT_M,Obj,GAIA,-,False,0.0,1.15,-13.11,1.44,0,1.51,-7.57,1.8,0,-14.31,0.99,-15.2,1.08,0,-9999.0,-9999.0,-9999.0,-9999.0,False,0,False,False,70.0,9,False,5607016,220034
http://www.lamost.org/dr8/v2.0//medspectrum/fits/681112046,J225014.39+433935.9,681112046,342.55997,43.659987,G14220639911232,1935439776865267072,68111204684108789R,2018-10-17T00:00:00.0,R,58409,46.68,58408,-11.29,NT224405N414909M01,-8.68,12,46,84108789,342.55997,43.659987,1935439776865267072,12.9221,13.2615,12.4196,MRS_NT_M,Obj,GAIA,-,False,0.0,1.0,-11.54,1.32,0,1.47,-10.81,1.76,0,-10.16,0.89,-11.05,1.0,0,-9999.0,-9999.0,-9999.0,-9999.0,False,0,False,False,69.9,9,False,5611013,220034
http://www.lamost.org/dr8/v2.0//medspectrum/fits/681112046,J225014.39+433935.9,681112046,342.55997,43.659987,G14220639911232,1935439776865267072,68111204684108813R,2018-10-17T00:00:00.0,R,58409,48.06,58408,-7.69,NT224405N414909M01,-12.77,12,46,84108813,342.55997,43.659987,1935439776865267072,12.9221,13.2615,12.4196,MRS_NT_M,Obj,GAIA,-,False,0.0,1.03,-7.94,1.34,0,1.38,-14.9,1.69,0,-9.18,0.94,-10.07,1.04,0,-9999.0,-9999.0,-9999.0,-9999.0,False,0,False,False,69.7,9,False,5611111,220034
http://www.lamost.org/dr8/v2.0//medspectrum/fits/681112046,J225014.39+433935.9,681112046,342.55997,43.659987,G14220639911232,1935439776865267072,68111204684108813B,2018-10-17T00:00:00.0,B,58409,28.18,58408,-7.69,NT224405N414909M01,-12.77,12,46,84108813,342.55997,43.659987,1935439776865267072,12.9221,13.2615,12.4196,MRS_NT_M,Obj,GAIA,-,False,0.0,1.03,-7.94,1.34,0,1.38,-14.9,1.69,0,-9.18,0.94,-10.07,1.04,0,-11.59,0.8,-11.55,0.84,False,0,False,False,69.7,9,False,5616275,220034
http://www.lamost.org/dr8/v2.0//medspectrum/fits/681112046,J225014.39+433935.9,681112046,342.55997,43.659987,G14220639911232,1935439776865267072,68111204684108789B,2018-10-17T00:00:00.0,B,58409,29.96,58408,-11.29,NT224405N414909M01,-8.68,12,46,84108789,342.55997,43.659987,1935439776865267072,12.9221,13.2615,12.4196,MRS_NT_M,Obj,GAIA,-,False,0.0,1.0,-11.54,1.32,0,1.47,-10.81,1.76,0,-10.16,0.89,-11.05,1.0,0,-12.0,0.83,-11.96,0.87,False,0,False,False,69.9,9,False,5616695,220034
http://www.lamost.org/dr8/v2.0//medspectrum/fits/681112046,J225014.39+433935.9,681112046,342.55997,43.659987,G14220639911232,19354

In [52]:
result[0]['product_url'][-1]

'http://www.lamost.org/dr8/v2.0//medspectrum/fits/681112046'

In [47]:
import requests

url = result[0]['product_url'][0] #choose the first spectra

data = requests.get(url, timeout=30)
with open("lamost_spectrum.fits", "wb") as f:
    f.write(data.content)

In [48]:
from astropy.io import fits
hdul = fits.open("lamost_spectrum.fits")
hdul.info()

wave = hdul[1].data['wavelength']
flux = hdul[1].data['flux']

Filename: lamost_spectrum.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  Information    1 PrimaryHDU      85   ()      
  1  COADD_B       1 BinTableHDU     37   1R x 6C   [4175E, 4175E, 4175E, 4175I, 4175I, 4175E]   
  2  COADD_R       1 BinTableHDU     37   1R x 6C   [3710E, 3710E, 3710E, 3710I, 3710I, 3710E]   
  3  B-84108765    1 BinTableHDU     36   1R x 5C   [4136E, 4136E, 4136E, 4136I, 4136E]   
  4  B-84108789    1 BinTableHDU     36   1R x 5C   [4136E, 4136E, 4136E, 4136I, 4136E]   
  5  B-84108813    1 BinTableHDU     36   1R x 5C   [4136E, 4136E, 4136E, 4136I, 4136E]   
  6  R-84108765    1 BinTableHDU     36   1R x 5C   [4136E, 4136E, 4136E, 4136I, 4136E]   
  7  R-84108789    1 BinTableHDU     36   1R x 5C   [4136E, 4136E, 4136E, 4136I, 4136E]   
  8  R-84108813    1 BinTableHDU     36   1R x 5C   [4136E, 4136E, 4136E, 4136I, 4136E]   
